# Fibromyalgia RAG Pipeline (v2 - layout-aware parsing)

A Retrieval-Augmented Generation (RAG) pipeline built over a single biomedical
review article: *"Fibromyalgia: A Review of the Pathophysiological Mechanisms
and Multidisciplinary Treatment Strategies"* (Jurado-Priego et al., 2024,
*Biomedicines* 12, 1543).

**What changed from v1, and why**

The previous version parsed sections with `text.find("2. Epidemiology")` on a
whitespace-collapsed string, and dropped the reference list entirely. That
works only for this one article's exact heading text and throws away every
citation. This version instead:

1. Extracts **PDF blocks with font metadata** (font family, size) instead of
   plain text, so numbered headings are detected by *how they're
   typeset* (bold for `N.`, italic for `N.N.`) rather than by hardcoded
   strings. This generalizes to any MDPI-style review article with the same
   numbering convention, not just this one PDF.
2. **Parses the reference list into structured, numbered entries** instead of
   discarding it, so an in-text marker like `[12,45]` can be resolved back to
   the actual source at answer-generation time.
3. **Chunks by sentence, not by raw character count**, and protects
   `[12,45]`-style citation markers during splitting so a chunk never ends
   mid-citation.
4. Fixes hyphenation rejoining at the correct point in the pipeline (the v1
   regex ran too late to ever match anything, so words like
   "musculoskeletal" stayed broken as "muscu- loskeletal").

**Pipeline stages**

1. Extract PDF lines with layout/font metadata.
2. Detect numbered section headings from typography.
3. Build section text (with hyphenation rejoined at the right point).
4. Parse the reference list into addressable entries.
5. Chunk each section sentence-by-sentence, protecting citation markers.
6. Wrap chunks as `Document`s with section + citation metadata.
7. Embed the chunks and index them in a FAISS vector store.
8. Retrieve the top-K chunks for a question and score retrieval quality with
   a small Precision@K keyword benchmark.
9. Generate a grounded answer from the retrieved chunks - via the OpenAI API
   if a key is configured, or via a template-based extractive fallback
   otherwise - resolving any cited reference numbers back to their full
   citation text.

This notebook mirrors the library implementation in
[`src/pipeline.py`](../src/pipeline.py), which is what the Gradio app
(`src/app.py`) actually imports; this notebook is the documented,
step-by-step walkthrough of the same logic.

## 1. Setup

In [ ]:
%pip install -q pymupdf langchain-core langchain-text-splitters \
    langchain-community langchain-huggingface faiss-cpu sentence-transformers openai


In [ ]:
import json
import os
import re
from pathlib import Path

# Project-relative paths (works locally, in Colab, and in CI alike)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = DATA_RAW_DIR / "biomedicines-12-01543.pdf"

assert PDF_PATH.exists(), (
    f"Source PDF not found at {PDF_PATH}. "
    "Place 'biomedicines-12-01543.pdf' in data/raw/ before running this notebook."
)
print("Using PDF:", PDF_PATH)


## 2. Layout-aware extraction

Instead of `page.get_text()` (a flat string with all layout information
discarded), we pull PyMuPDF's `"dict"` output, which gives every line its
**font name, size, and position**. This is what makes heading detection in
Section 3 possible without hardcoding heading strings. Running headers/
footers and "`N of 22`" page-number markers are filtered out here, at the
line level, so they can never leak into a later chunk.

In [ ]:
import fitz  # PyMuPDF

JUNK_LINE_PATTERNS = [
    r'^Biomedicines\s+\d{4},\s*\d+,?\s*(x FOR PEER REVIEW|\d+)$',  # repeated citation banner
    r'^\d+\s+of\s+\d+$',                                             # "4 of 22" page markers
]

def extract_lines(pdf_path: Path):
    """Extract text lines with font metadata (layout-aware, not plain text)."""
    doc = fitz.open(pdf_path)
    lines = []
    for pno, page in enumerate(doc):
        d = page.get_text("dict")
        for block in d["blocks"]:
            if block["type"] != 0:  # skip images
                continue
            for line in block["lines"]:
                spans = line["spans"]
                if not spans:
                    continue
                text = "".join(s["text"] for s in spans).strip()
                if not text:
                    continue
                lines.append({
                    "page": pno,
                    "text": text,
                    "fonts": {s["font"] for s in spans},
                    "y": line["bbox"][1],
                })
    doc.close()
    return [l for l in lines if not any(re.match(p, l["text"]) for p in JUNK_LINE_PATTERNS)]

lines = extract_lines(PDF_PATH)
print(f"Extracted {len(lines)} text lines (after removing running headers/footers)")
print(lines[5])


## 3. Heading Detection

Numbered headings in this article follow a consistent typographic
convention (verified against the actual PDF fonts, not assumed):

| Level | Pattern | Font |
|---|---|---|
| 1 | `N. Title` | Bold |
| 2 | `N.N. Title` | Italic |
| 3 | `N.N.N. Title` | Regular, but on its own line |

Detecting headings this way means the parser doesn't need to know the
article's actual section names in advance -- it will find `8. New Section`
in a different paper just as reliably as `2. Epidemiology` here.

In [ ]:
H1 = re.compile(r'^(\d{1,2})\.\s+(.+)$')
H2 = re.compile(r'^(\d{1,2}\.\d{1,2})\.\s+(.+)$')
H3 = re.compile(r'^(\d{1,2}\.\d{1,2}\.\d{1,2})\.\s+(.+)$')

def _is_bold(block):
    return any('Bold' in f for f in block["fonts"])

def _is_italic(block):
    return any('Ital' in f for f in block["fonts"])

def detect_headings(lines):
    headings = []
    for i, b in enumerate(lines):
        t = b["text"]
        m3 = H3.match(t)
        m2 = H2.match(t) if not m3 else None
        m1 = H1.match(t) if not (m2 or m3) else None
        if m3:
            headings.append({"level": 3, "number": m3.group(1), "title": m3.group(2), "line_idx": i})
        elif m2 and _is_italic(b):
            headings.append({"level": 2, "number": m2.group(1), "title": m2.group(2), "line_idx": i})
        elif m1 and _is_bold(b):
            headings.append({"level": 1, "number": m1.group(1), "title": m1.group(2), "line_idx": i})
    return headings

headings = detect_headings(lines)
print(f"Detected {len(headings)} headings\n")
for h in headings:
    print(" " * ((h["level"] - 1) * 3), h["number"], h["title"])


## 4. Section Building

Slice the line stream between consecutive headings. Hyphenation rejoining
(`"muscu-" + "loskeletal"` -> `"musculoskeletal"`) happens **at join time**,
which is the only point where the original `-` + line-break pattern is still
visible -- doing it after lines are already space-joined (as v1 did) can
never match anything.

In [ ]:
def join_lines_dehyphenated(text_lines) -> str:
    out = ""
    for line in text_lines:
        if out.endswith("-") and line and line[0].islower():
            out = out[:-1] + line          # rejoin split word, no space
        elif out:
            out = out + " " + line
        else:
            out = line
    return out

def build_sections(lines, headings):
    sections = []
    for i, h in enumerate(headings):
        start = h["line_idx"] + 1
        end = headings[i + 1]["line_idx"] if i + 1 < len(headings) else len(lines)
        body_lines = [l["text"] for l in lines[start:end]]
        text = re.sub(r'\s+', ' ', join_lines_dehyphenated(body_lines)).strip()
        sections.append({
            "number": h["number"],
            "level": h["level"],
            "title": h["title"],
            "top_section": h["number"].split(".")[0],
            "text": text,
        })
    return sections

sections = build_sections(lines, headings)
for s in sections[:5]:
    print(f"[{s['number']:>6s}] {s['title']:<45s} -> {len(s['text']):5,d} chars")
print("...")
print(f"\nTotal sections/subsections: {len(sections)}")


## 5. Reference-List Parsing

The v1 pipeline discarded everything after "References". Here the reference
list is parsed into individually addressable entries, keyed by citation
number, so any `[N]` marker found in a chunk can be resolved back to its
source at answer-generation time.

Note the entry-boundary regex requires a letter (not just an uppercase
letter) after the number, because some author surnames start lowercase
(e.g. "da Rocha, A.P...."). This was caught by testing against the real
reference list, not assumed.

In [ ]:
REF_ENTRY = re.compile(r'\n(\d{1,3})\.\s+(?=[A-Za-z])')

def parse_references(pdf_path: Path) -> dict:
    doc = fitz.open(pdf_path)
    raw_text = "".join(page.get_text() + "\n" for page in doc)
    doc.close()

    m = re.search(r'\nReferences\n', raw_text)
    if not m:
        return {}
    ref_text = raw_text[m.end():]
    ref_text = ref_text.split("Disclaimer/Publisher")[0]
    ref_text = re.sub(r'Biomedicines\s+\d{4},\s*\d+,?\s*\d+\s*\n?\d*\s*of\s*\d+\s*\n?', '', ref_text)
    ref_text = re.sub(r'\n\d+\s+of\s+\d+\n', '\n', ref_text)

    matches = list(REF_ENTRY.finditer("\n" + ref_text))
    entries = {}
    for i, mm in enumerate(matches):
        num = int(mm.group(1))
        start = mm.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(ref_text) + 1
        content = ("\n" + ref_text)[start:end]
        entries[num] = re.sub(r'\s+', ' ', content).strip()
    return entries

references = parse_references(PDF_PATH)
print(f"Parsed {len(references)} reference entries")
if references:
    sample_num = sorted(references)[0]
    print(f"\nExample [{sample_num}]:", references[sample_num][:140])


## 6. Citation-Aware Chunking

Sentences are split without ever cutting inside a `[12,45]` marker (commas
inside brackets are temporarily masked before splitting, then restored).
Chunks are built up sentence-by-sentence to a soft character cap, so a
sentence -- and therefore a citation -- is never split across two chunks.
Each chunk also carries the list of reference numbers it actually cites.

In [ ]:
CITATION_MARKER = re.compile(r'\[([\d,\s]+)\]')

def split_sentences_keep_citations(text: str):
    protected = re.sub(r'\[([\d,\s\-]+)\]',
                        lambda mm: '[' + mm.group(1).replace(',', '\u00a7') + ']', text)
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', protected)
    return [s.replace('\u00a7', ',') for s in sentences]

def extract_citation_numbers(text: str):
    nums = set()
    for m in CITATION_MARKER.finditer(text):
        for n in m.group(1).split(','):
            n = n.strip()
            if n.isdigit():
                nums.add(int(n))
    return sorted(nums)

def chunk_section(section: dict, max_chars: int = 900):
    sentences = split_sentences_keep_citations(section["text"])
    chunks, current = [], ""
    for sent in sentences:
        if current and len(current) + len(sent) + 1 > max_chars:
            chunks.append(current.strip())
            current = sent
        else:
            current = f"{current} {sent}".strip()
    if current:
        chunks.append(current.strip())
    return [
        {
            "text": c,
            "section_number": section["number"],
            "section_title": section["title"],
            "top_section": section["top_section"],
            "level": section["level"],
            "cited_refs": extract_citation_numbers(c),
        }
        for c in chunks
    ]

all_chunks = []
for s in sections:
    all_chunks.extend(chunk_section(s))

print(f"Total chunks: {len(all_chunks)}")
with_citations = [c for c in all_chunks if c["cited_refs"]]
print(f"Chunks carrying at least one citation: {len(with_citations)} / {len(all_chunks)}")
if with_citations:
    sample = with_citations[0]
    print(f"\nSample chunk (section {sample['section_number']}, cites {sample['cited_refs']}):")
    print(sample["text"][:300])


## 7. Building Documents

Each chunk is wrapped as a langchain `Document`, combining the article-level
metadata (title, authors, journal) with the chunk's section and citation
metadata. A regression check confirms no leftover "`N of 22`" page-number
artifacts survived extraction and chunking.

In [ ]:
from langchain_core.documents import Document

METADATA = {
    "title": (
        "Fibromyalgia: A Review of the Pathophysiological Mechanisms and "
        "Multidisciplinary Treatment Strategies"
    ),
    "authors": [
        "Lina Noelia Jurado-Priego",
        "Cristina Cueto-Ureña",
        "María Jesús Ramírez-Expósito",
        "José Manuel Martínez-Martos",
    ],
    "source": PDF_PATH.name,
    "journal": "Biomedicines",
    "year": 2024,
}

documents = [
    Document(
        page_content=chunk["text"],
        metadata={
            **METADATA,
            "section": f"{chunk['section_number']}. {chunk['section_title']}",
            "section_number": chunk["section_number"],
            "section_title": chunk["section_title"],
            "top_section": chunk["top_section"],
            "level": chunk["level"],
            "cited_refs": chunk["cited_refs"],
        },
    )
    for chunk in all_chunks
]

bad_docs = [d for d in documents if re.search(r'\b\d+\s+of\s+\d+\b', d.page_content)]
print(f"Documents still containing page-number artifacts: {len(bad_docs)}")
assert len(bad_docs) == 0, "Text extraction regression: page numbers leaked into chunks"

print(f"Built {len(documents)} chunk-level documents")
documents[0]


## 8. Embedding & Indexing

Embed every chunk with a small sentence-transformer model and index the
vectors in a FAISS store for similarity search.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Indexing complete:", vectorstore.index.ntotal, "vectors")


In [ ]:
# Persist the index (and the parsed reference list) so they don't need to
# be rebuilt on every run.
INDEX_DIR = DATA_PROCESSED_DIR / "faiss_index"
vectorstore.save_local(str(INDEX_DIR))

REFERENCES_PATH = DATA_PROCESSED_DIR / "references.json"
REFERENCES_PATH.write_text(
    json.dumps({str(k): v for k, v in references.items()}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved FAISS index to", INDEX_DIR)
print("Saved reference list to", REFERENCES_PATH)


## 9. Retrieval Evaluation

A lightweight Precision@K benchmark: for each test question, check whether
any of the top-K retrieved chunks contain at least one of the expected
keywords.

In [ ]:
def evaluate_retrieval(eval_dataset, retriever) -> float:
    relevant_count = 0
    for item in eval_dataset:
        docs = retriever.invoke(item["question"])
        retrieved_text = " ".join(d.page_content for d in docs)
        if any(kw.lower() in retrieved_text.lower() for kw in item["keywords"]):
            relevant_count += 1
    score = (relevant_count / len(eval_dataset)) * 100
    print(f"Retrieval Precision@K Score: {score:.2f}%")
    return score

eval_dataset = [
    {
        "question": "What are the FDA-approved drugs for fibromyalgia?",
        "keywords": ["pregabalin", "duloxetine", "milnacipran"],
    },
    {
        "question": "What is fibromyalgia characterized by?",
        "keywords": ["chronic", "widespread pain", "fatigue"],
    },
    {
        "question": "What diagnostic criteria are used for fibromyalgia?",
        "keywords": ["tender points", "ACR", "criteria"],
    },
]

evaluate_retrieval(eval_dataset, retriever)


In [ ]:
query = "What are the FDA-approved drugs for fibromyalgia?"
results = retriever.invoke(query)

print(f"Query: {query}\n" + "=" * 60)
for i, doc in enumerate(results, 1):
    print(f"\n[Chunk {i}] section={doc.metadata['section']} cites={doc.metadata['cited_refs']}")
    print(doc.page_content.strip())
    print("-" * 60)


## 10. Answer Generation (RAG)

If an `OPENAI_API_KEY` environment variable is set, the retrieved chunks are
passed to an OpenAI chat model to synthesize a grounded answer. If no key is
configured (e.g. in CI, or when running this notebook without any paid API),
the pipeline falls back to a template-based extractive answer built directly
from the retrieved chunks, so the notebook remains fully runnable end-to-end
without needing any secret. Either way, any reference numbers cited by the
retrieved chunks are resolved back to their full citation text and appended
to the answer.

In [ ]:
def generate_answer(question: str, retriever, model: str = "gpt-4o-mini", references: dict | None = None) -> str:
    """Answer `question` using retrieved context (RAG).

    Uses the OpenAI API when OPENAI_API_KEY is set; otherwise falls back to a
    simple extractive summary built from the retrieved chunks so the notebook
    still runs end-to-end without any API key or network access. Cited
    reference numbers are resolved back to their full text when `references`
    is provided.
    """
    docs = retriever.invoke(question)
    context = "\n\n".join(
        f"[Section: {d.metadata['section']}]\n{d.page_content}" for d in docs
    )

    api_key = os.environ.get("OPENAI_API_KEY")
    if api_key:
        from openai import OpenAI

        client = OpenAI(api_key=api_key)
        system_prompt = (
            "You are a biomedical research assistant. Answer the question "
            "using ONLY the provided context from the article. If the answer "
            "is not contained in the context, say so explicitly. Cite the "
            "section name(s) you drew on."
        )
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
            ],
            temperature=0,
        )
        answer = response.choices[0].message.content
    else:
        sections_used = ", ".join(sorted({d.metadata["section"] for d in docs}))
        preview = "\n\n".join(d.page_content.strip() for d in docs)
        answer = (
            f"[Extractive fallback - no OPENAI_API_KEY set]\n"
            f"Most relevant sections: {sections_used}\n\n"
            f"Retrieved context:\n{preview}"
        )

    if references:
        cited_nums = sorted({n for d in docs for n in d.metadata.get("cited_refs", [])})
        citation_lines = [f"[{n}] {references[n]}" for n in cited_nums if n in references]
        if citation_lines:
            answer = f"{answer}\n\nReferences cited in the retrieved passages:\n" + "\n".join(citation_lines)

    return answer

answer = generate_answer(
    "What non-pharmacological treatments are discussed for fibromyalgia?",
    retriever,
    references=references,
)
print(answer)


## 11. Results

- Numbered section headings were detected from PDF layout metadata rather
  than a hardcoded heading list, so the parser generalizes to other
  MDPI-style review articles.
- The reference list was parsed into individually addressable entries so
  in-text citation markers can be resolved back to their source.
- Chunking is sentence-aware and citation-aware: no chunk ends mid-sentence
  or mid-citation-marker, and each chunk records the reference numbers it
  cites.
- Retrieval Precision@K on the 3-question benchmark: see the printed score
  in Section 9 (this benchmark is small and intended as a smoke test, not a
  statistically rigorous evaluation).
- The RAG loop is complete end-to-end: retrieval results feed into an answer
  generation step (Section 10) that also resolves cited references back to
  their full text.

## 12. Conclusion

This notebook implements a minimal, reproducible, layout-aware RAG pipeline
over a single biomedical review article - PDF -> layout-aware extraction ->
heading detection -> section building -> reference parsing -> citation-aware
chunking -> embeddings -> retrieval -> grounded generation with citation
resolution.

## 13. Limitations

- The corpus is a single article; the retrieval evaluation is a 3-question
  smoke test, not a statistically powered benchmark.
- Heading detection assumes the `N.` (bold) / `N.N.` (italic) / `N.N.N.`
  numbering convention used by this article; a differently typeset article
  would need its own font/style rules.
- Full LLM-based generation requires an OpenAI API key (not included, and
  never committed to this repository). Without one, generation degrades to
  an extractive summary of the retrieved context.
- No automated evaluation (e.g. faithfulness or answer-relevance scoring) is
  performed on the generation step itself - only on retrieval.